This notebook joins all of the datasets together and cleans up/ averages the player stats. 

In [281]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk

In [282]:
# Read in the data
recruit_scores = pd.read_csv(r'../Data/qb_trait_scores.csv')

In [283]:
# Examine null values per column
recruit_scores.isna().sum()

player                        0
url                           0
rank                          0
height                        0
weight                        0
composite_rating              0
school_info                   0
city                          0
state                         0
draft_projection              9
reminds_of                   36
evaluated_date                0
analyst                      15
athletic_background           0
committed_school              0
numerical_rating              0
Is_FBS                        0
Is_SEC                        0
Is_Big_Ten                    0
Is_Big_XII                    0
Is_ACC                        0
pos_DUAL                      0
pos_PRO                       0
pos_QB                        0
star_2                        0
star_3                        0
star_4                        0
star_5                        0
Is_Top_Five_Ranked            0
Is_From_East                  0
Is_From_West                  0
weight_t

Note lot of prospects are missing certain trait categories. Specifically Football IQ, Decision Making and Leadership & Intangibles. These columns will be dropped and because of the much lower amount of missingness in the other columns, any players with missing data in the following will be dropped: Accuracy, Arm Strength, Mobility & Athleticism, Pocket Presence, Size & Physical

In [284]:
# Drop columns with lots of missing info
recruit_scores = recruit_scores.drop(columns=['Football IQ', 'Decision Making', 'Leadership & Intangibles'])


# Drop rows with missing values in the indicated columns
recruit_scores = recruit_scores.dropna(subset=['Accuracy', 'Arm Strength', 'Mobility & Athleticism', 'Pocket Presence', 'Size & Physical'])

In [285]:
# Check new results of null values per column
recruit_scores.isna().sum()

player                     0
url                        0
rank                       0
height                     0
weight                     0
composite_rating           0
school_info                0
city                       0
state                      0
draft_projection           8
reminds_of                20
evaluated_date             0
analyst                   10
athletic_background        0
committed_school           0
numerical_rating           0
Is_FBS                     0
Is_SEC                     0
Is_Big_Ten                 0
Is_Big_XII                 0
Is_ACC                     0
pos_DUAL                   0
pos_PRO                    0
pos_QB                     0
star_2                     0
star_3                     0
star_4                     0
star_5                     0
Is_Top_Five_Ranked         0
Is_From_East               0
Is_From_West               0
weight_to_height_ratio     0
bmi                        0
Accuracy                   0
Arm Strength  

### Let's look at player stats_2018_2025


ultimately, we want to look at the success players achieve at the original school they were recruited to. To do this, we will id players who have at least 2 years at the school they were recruited to and aggregated their stats together. This way we can train out models on multiple years of data for a player.

In [286]:
# Read in the data
qb_stats = pd.read_csv(r'../Data/player_stats_2018_2025.csv', encoding='latin1')

In [287]:
qb_stats[qb_stats['Player'] == 'Jayden Daniels']

,Rk,Player,Team,Conf,G,Cmp,Att,Cmp%,Yds,TD,TD%,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,Awards,Year
1128,129,Jayden Daniels,Arizona State,Pac-12,4,12.3,21.0,58.3,175.3,1.25,6.0,0.25,1.2,8.3,9.0,14.3,175.3,145.7,NaN,2020


In [288]:
# look at the data output
qb_stats

,Rk,Player,Team,Conf,G,Cmp,Att,Cmp%,Yds,TD,TD%,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,Awards,Year
0,1,Dwayne Haskins*,Ohio State,Big Ten,14,26.6,38.10,70.0,345.1,3.57,9.4,0.57,1.5,9.1,10.26,13.0,345.1,174.1,H-3,2018
1,2,Gardner Minshew*,Washington State,Pac-12,13,36.0,50.90,70.7,367.6,2.92,5.7,0.69,1.4,7.2,7.76,10.2,367.6,147.6,H-5,2018
2,3,Kyler Murray*,Oklahoma,Big 12,14,18.6,26.90,69.0,311.5,3.00,11.1,0.50,1.9,11.6,12.96,16.8,311.5,199.2,H-1,2018
3,4,Taylor Cornelius*,Oklahoma State,Big 12,13,22.2,37.30,59.4,306.0,2.46,6.6,1.00,2.7,8.2,8.32,13.8,306.0,144.7,NaN,2018
4,5,Tua Tagovailoa*,Alabama,SEC,15,16.3,23.70,69.0,264.4,2.87,12.1,0.40,1.7,11.2,12.83,16.2,264.4,199.4,"H-2,Maxwell,AA,Camp",2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3931,496,Gavin Freeman,Oklahoma State,Big 12,12,0.0,0.08,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,NaN,0.0,0.0,NaN,2025
3932,497,Grady Gross*,Washington,Big Ten,13,0.0,0.08,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,NaN,0.0,0.0,NaN,2025
3933,498,Grant Page*,Southern Mississippi,Sun Belt,11,0.0,0.09,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,NaN,0.0,0.0,NaN,2025
3934,499,Holden Geriner*,Texas State,Sun Belt,3,0.0,0.33,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,NaN,0.0,0.0,NaN,2025


In [289]:
# Take out all asterisks from player names
qb_stats['Player'] = qb_stats['Player'].str.replace('*', '', regex=False)

In [290]:
# Examine how many entries there are for each player/ team combo
qb_stats.groupby(['Player','Team']).size().sort_values(ascending=False)

Player            Team          
Chandler Fields   Louisiana         6
Jalon Daniels     Kansas            6
Gunnar Watson     Troy              6
Brett Gabbert     Miami (OH)        6
Tyler Page        SMU               6
                                   ..
Zeb Noland        South Carolina    1
                  Iowa State        1
Zavion Thomas     LSU               1
Zane Flores       Oklahoma State    1
Zakhari Franklin  UTSA              1
Length: 2453, dtype: int64

In [291]:
qb_stats.columns

Index(['Rk', 'Player', 'Team', 'Conf', 'G', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD',
       'TD%', 'Int', 'Int%', 'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate', 'Awards',
       'Year'],
      dtype='str')

note: columns that will be averaged across seasons are 'G', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'TD%', 'Int', 'Int%', 
             'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate'

the column that will be kept the same are the 'Player', 'Team', and 'Conf' columns. Note: keep_cols below only contains 'Conf' because the 'Player' and 'Team' combos are already implemented by the .groupby() function. 

In [292]:
# Find players with multiple rows but were with the same team (indicating multiple years of stats for that player/ team combo)
multi_year_players = qb_stats.groupby(['Player','Team']).filter(lambda x: len(x) > 1)

# Average the stat columns, keep first value of categorical columns
stat_cols = ['G', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'TD%', 'Int', 'Int%', 
             'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate']
keep_cols = ['Conf']

averaged_df = (
    multi_year_players
    .groupby(['Player','Team'])
    .agg({**{col: 'mean' for col in stat_cols},
          **{col: 'first' for col in keep_cols}})
    .reset_index()
)

# Reorder columns to match original: Player, Team, Conf, stats...
averaged_df = averaged_df[['Player', 'Team', 'Conf'] + stat_cols]

In [293]:
# Check row count after averaging
averaged_df.shape

(928, 17)

In [294]:
averaged_df

,Player,Team,Conf,G,Cmp,Att,Cmp%,Yds,TD,TD%,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate
0,AJ Bianco,Nevada,MWC,7.666667,3.843333,6.533333,60.533333,43.40,0.266667,3.766667,0.243333,2.766667,6.30,5.793333,10.466667,43.40,120.233333
1,AJ Hairston,Massachusetts,Ind,7.000000,10.350000,19.950000,52.150000,107.85,0.825000,4.350000,0.200000,0.950000,5.50,5.960000,10.450000,107.85,110.900000
2,AJ Mayer,Miami (OH),MAC,5.000000,8.690000,17.100000,50.650000,122.30,1.095000,6.500000,0.310000,1.800000,7.15,7.615000,14.100000,122.30,128.350000
3,AJ Padgett,Rice,CUSA,4.500000,12.400000,22.600000,55.650000,165.75,1.500000,6.650000,0.665000,3.200000,7.20,7.085000,13.250000,165.75,131.700000
4,AJ Swann,Vanderbilt,SEC,7.500000,15.300000,27.350000,56.350000,192.20,1.555000,5.600000,0.695000,2.300000,6.90,7.020000,12.350000,192.20,128.450000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,Zamar Wise,Massachusetts,Ind,5.000000,1.095000,2.645000,50.000000,8.92,0.000000,0.000000,0.000000,0.000000,3.05,3.055000,7.350000,8.90,75.650000
924,Zeon Chriss,Houston,Big 12,8.500000,4.610000,7.650000,55.700000,46.45,0.265000,3.950000,0.450000,5.500000,5.70,4.065000,10.350000,46.45,105.950000
925,Zeon Chriss,Louisiana,Sun Belt,6.000000,7.525000,11.425000,63.350000,90.05,0.815000,6.950000,0.440000,5.000000,7.65,6.775000,12.050000,90.05,140.300000
926,Zevi Eckhaus,Washington State,Ind,7.000000,14.900000,22.300000,68.450000,165.85,1.345000,6.450000,0.880000,3.950000,7.65,7.160000,11.150000,165.85,146.050000


We also want to save information regarding how many seasons each player played for a team. We can simply do this by saving the count from earlier and assigning it to each player

In [295]:
# save into dataframe
qb_counts = pd.DataFrame(qb_stats.groupby(['Player','Team']).size().reset_index(name='Counts'))

In [296]:
# look at results
qb_counts

,Player,Team,Counts
0,A.J. Davis,UAB,1
1,A.J. Erdely,UAB,1
2,AJ Bianco,Nevada,3
3,AJ Bush,Illinois,1
4,AJ Duffy,Florida State,1
...,...,...,...
2448,Zeon Chriss,Louisiana,2
2449,Zevi Eckhaus,Washington State,2
2450,Zion Turner,Connecticut,2
2451,Zion Turner,Marshall,1


In [297]:
# test an example player
qb_counts[qb_counts['Player'] == 'Sam Hartman']

,Player,Team,Counts
2069,Sam Hartman,Notre Dame,1
2070,Sam Hartman,Wake Forest,5


In [298]:
# Merge the averaged stats with the counts of entries per player/ team combo
merged_df = pd.merge(averaged_df, qb_counts, on=['Player', 'Team'], how='inner')

In [299]:
merged_df[merged_df['Player'] == 'Sam Hartman']

,Player,Team,Conf,G,Cmp,Att,Cmp%,Yds,TD,TD%,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,Counts
776,Sam Hartman,Wake Forest,ACC,9.6,18.66,31.78,58.44,257.08,2.036,6.2,0.79,2.44,8.08,8.236,13.84,257.08,142.0,5


Note: Notre Dame Sam Hartman was removed from the data during the average_df creation because he only played 1 year there.

With this, lets merge this data with the recruit_scores data.

In [300]:
recruit_scores.columns

Index(['player', 'url', 'rank', 'height', 'weight', 'composite_rating',
       'school_info', 'city', 'state', 'draft_projection', 'reminds_of',
       'evaluated_date', 'analyst', 'athletic_background', 'committed_school',
       'numerical_rating', 'Is_FBS', 'Is_SEC', 'Is_Big_Ten', 'Is_Big_XII',
       'Is_ACC', 'pos_DUAL', 'pos_PRO', 'pos_QB', 'star_2', 'star_3', 'star_4',
       'star_5', 'Is_Top_Five_Ranked', 'Is_From_East', 'Is_From_West',
       'weight_to_height_ratio', 'bmi', 'Accuracy', 'Arm Strength',
       'Mobility & Athleticism', 'NFL / Draft Projection', 'Pocket Presence',
       'Size & Physical'],
      dtype='str')

In [301]:
merged_df.columns

Index(['Player', 'Team', 'Conf', 'G', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'TD%',
       'Int', 'Int%', 'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate', 'Counts'],
      dtype='str')

In [302]:
# merge the data
players = pd.merge(recruit_scores, merged_df, left_on=['player','committed_school'], right_on=['Player', 'Team'], how='inner')

In [303]:
players.shape

(48, 57)

In [304]:
players

,player,url,rank,height,weight,composite_rating,school_info,city,state,draft_projection,...,TD,TD%,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,Counts
0,Arch Manning,https://247sports.com/player/arch-manning-4608...,1,76.0,215,100,"Isidore Newman (New Orleans, LA)",New Orleans,LA,Round 1 - Top 10,...,0.966667,5.466667,0.246667,1.300000,8.066667,8.590000,14.400000,117.400000,139.766667,3
1,Athan Kaliakmanis,https://247sports.com/player/athan-kaliakmanis...,26,76.0,205,90,"Antioch (Antioch, IL)",Antioch,IL,4-7 Round - Day 3,...,0.720000,3.750000,0.555000,3.350000,7.400000,6.635000,13.800000,119.600000,121.300000,2
2,Austin Novosad,https://247sports.com/player/austin-novosad-46...,13,75.0,185,94,"Dripping Springs (Dripping Springs, TX)",Dripping Springs,TX,2-3 Round - Day 2,...,0.000000,0.000000,0.000000,0.000000,9.400000,9.410000,16.433333,19.866667,156.300000,3
3,Avery Johnson,https://247sports.com/player/avery-johnson-461...,9,74.0,175,95,"Maize (Maize, KS)",Maize,KS,4-7 Round - Day 3,...,1.350000,6.533333,0.423333,1.500000,7.200000,7.816667,12.366667,155.766667,136.966667,3
4,Beau Allen,https://247sports.com/player/beau-allen-46039461/,64,74.0,203,85,"Lexington Catholic (Lexington, KY)",Lexington,KY,Power 4 Starter,...,0.000000,0.000000,0.000000,0.000000,6.700000,6.690000,12.400000,25.350000,111.000000,2
5,Behren Morton,https://247sports.com/player/behren-morton-460...,11,74.0,185,94,"Eastland (Eastland, TX)",Eastland,TX,4-7 Round - Day 3,...,1.272000,4.320000,0.528000,1.960000,5.640000,5.612000,9.060000,161.880000,113.960000,5
6,Brian Maurer,https://247sports.com/player/brian-maurer-4603...,24,75.0,199,88,"West Port (Ocala, FL)",Ocala,FL,Power 4 Starter,...,0.125000,1.350000,0.315000,3.350000,4.350000,3.125000,10.000000,34.950000,74.300000,2
7,Brock Glenn,https://247sports.com/player/brock-glenn-46114...,21,74.0,195,90,"Lausanne Collegiate School (Covington, TN)",Covington,TN,4-7 Round - Day 3,...,0.356667,7.833333,0.370000,2.766667,5.533333,5.863333,11.766667,49.466667,114.200000,3
8,Brock Vandagriff,https://247sports.com/player/brock-vandagriff-...,5,75.0,205,97,"Prince Avenue Christian School (Bogart, GA)",Bogart,GA,Second Round,...,0.083333,3.700000,0.000000,0.000000,3.066667,3.796667,13.800000,6.866667,60.100000,3
9,Bryce Young,https://247sports.com/player/bryce-young-93127/,1,71.0,183,101,"Mater Dei (Santa Ana, CA)",Santa Ana,CA,Round 1 - Top 10,...,1.976667,7.166667,0.296667,0.866667,8.266667,9.300000,12.966667,207.200000,154.800000,3


Note: Seems like half of the guys in the recruit data are missing in the merged_df (player stats). This may indicate one of the following:

- Did not play multiple years at committed school 
- Data on the school in which they played was not in the college football reference data (lower level schools like d2 most likely)
- Player was not a top 300 recruit.

All of these come as limitations from their respective datasets: 
The recruit data was hevaily filtered to only include players with no missingness and with scouting reports from 247 sports. In the first place, 247 sports is limited in that it only has the top 300 recruits (although I am not worried about this aspect as this list encapsulates most players in the class). 

College Football reference data only contains the top 500 qbs from a given year.


Despite the lack of data that is generated (48 rows); I believe that the data within these rows are quality datapoints and should be examined.

Despite the note above, let's go ahead and merge the team stats to this data.

Note: The way this will work in our model is we train on the teams stats the year before the qb got there. 

This will be done by obtaining the qbs recruiting class year, creating a new column that decrements that number by 1, and then merge then team stats and standings data to the players data via inner merge on the decremented year and team name columns.

In [305]:
print(players['player'].tolist())

['Arch Manning', 'Athan Kaliakmanis', 'Austin Novosad', 'Avery Johnson', 'Beau Allen', 'Behren Morton', 'Brian Maurer', 'Brock Glenn', 'Brock Vandagriff', 'Bryce Young', 'C.J. Stroud', 'Cade Klubnik', 'Cade McNamara', 'Cameron Edge', 'Carson Beck', 'Christopher Vizzina', 'Devin Brown', 'Drew Allar', 'Dylan Lonergan', 'Garrett Nussmeier', 'Gavin Wimsatt', 'Graham Mertz', 'Gunner Stockton', 'Harrison Bailey', 'Haynes King', 'Heinrich Haarberg', 'Hudson Card', 'J.J. McCarthy', 'Jackson Arnold', 'Jalen Milroe', 'Jayden Daniels', 'Kedon Slovis', 'Kenny Minchey', 'Kyle McCord', 'Luke Altmyer', 'Marcel Reed', 'Nico Iamaleava', 'Noah Fifita', 'Preston Stone', 'Rickie Collins', 'Riley Leonard', 'Ryan Hilinski', 'Sam Howell', 'Spencer Rattler', 'Ty Simpson', 'Ty Thompson', 'Tyler Buchner', 'Zach Pyron']


In [306]:
# Scrapper did not obtain player class so manually entered class for each player
# Create a mapping of players to their classes
qb_recruiting_classes = [
    ("Arch Manning", 2023),
    ("Athan Kaliakmanis", 2021),
    ("Austin Novosad", 2023),
    ("Avery Johnson", 2023),
    ("Beau Allen", 2020),
    ("Behren Morton", 2021),
    ("Brian Maurer", 2019),
    ("Brock Glenn", 2023),
    ("Brock Vandagriff", 2021),
    ("Bryce Young", 2020),
    ("C.J. Stroud", 2020),
    ("Cade Klubnik", 2022),
    ("Cade McNamara", 2019),
    ("Cameron Edge", 2022),
    ("Carson Beck", 2020),
    ("Christopher Vizzina", 2023),
    ("Devin Brown", 2022),
    ("Drew Allar", 2022),
    ("Dylan Lonergan", 2023),
    ("Garrett Nussmeier", 2021),
    ("Gavin Wimsatt", 2021),
    ("Graham Mertz", 2019),
    ("Gunner Stockton", 2022),
    ("Harrison Bailey", 2020),
    ("Haynes King", 2020),
    ("Heinrich Haarberg", 2021),
    ("Hudson Card", 2020),
    ("J.J. McCarthy", 2021),
    ("Jackson Arnold", 2023),
    ("Jalen Milroe", 2021),
    ("Jayden Daniels", 2019),
    ("Kedon Slovis", 2019),
    ("Kenny Minchey", 2023),
    ("Kyle McCord", 2021),
    ("Luke Altmyer", 2021),
    ("Marcel Reed", 2023),
    ("Nico Iamaleava", 2023),
    ("Noah Fifita", 2022),
    ("Preston Stone", 2021),
    ("Rickie Collins", 2023),
    ("Riley Leonard", 2021),
    ("Ryan Hilinski", 2019),
    ("Sam Howell", 2019),
    ("Spencer Rattler", 2019),
    ("Ty Simpson", 2022),
    ("Ty Thompson", 2021),
    ("Tyler Buchner", 2021),
    ("Zach Pyron", 2022)
]

# Convert the list into a dataframe
qb_recruiting_classes_df = pd.DataFrame(qb_recruiting_classes, columns=['player', 'recruiting_class'])

In [307]:
# Merge the dataset with the recruiting class information
players = players.merge(qb_recruiting_classes_df, on='player', how='inner')

In [308]:
# Check the results
players

,player,url,rank,height,weight,composite_rating,school_info,city,state,draft_projection,...,TD%,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,Counts,recruiting_class
0,Arch Manning,https://247sports.com/player/arch-manning-4608...,1,76.0,215,100,"Isidore Newman (New Orleans, LA)",New Orleans,LA,Round 1 - Top 10,...,5.466667,0.246667,1.300000,8.066667,8.590000,14.400000,117.400000,139.766667,3,2023
1,Athan Kaliakmanis,https://247sports.com/player/athan-kaliakmanis...,26,76.0,205,90,"Antioch (Antioch, IL)",Antioch,IL,4-7 Round - Day 3,...,3.750000,0.555000,3.350000,7.400000,6.635000,13.800000,119.600000,121.300000,2,2021
2,Austin Novosad,https://247sports.com/player/austin-novosad-46...,13,75.0,185,94,"Dripping Springs (Dripping Springs, TX)",Dripping Springs,TX,2-3 Round - Day 2,...,0.000000,0.000000,0.000000,9.400000,9.410000,16.433333,19.866667,156.300000,3,2023
3,Avery Johnson,https://247sports.com/player/avery-johnson-461...,9,74.0,175,95,"Maize (Maize, KS)",Maize,KS,4-7 Round - Day 3,...,6.533333,0.423333,1.500000,7.200000,7.816667,12.366667,155.766667,136.966667,3,2023
4,Beau Allen,https://247sports.com/player/beau-allen-46039461/,64,74.0,203,85,"Lexington Catholic (Lexington, KY)",Lexington,KY,Power 4 Starter,...,0.000000,0.000000,0.000000,6.700000,6.690000,12.400000,25.350000,111.000000,2,2020
5,Behren Morton,https://247sports.com/player/behren-morton-460...,11,74.0,185,94,"Eastland (Eastland, TX)",Eastland,TX,4-7 Round - Day 3,...,4.320000,0.528000,1.960000,5.640000,5.612000,9.060000,161.880000,113.960000,5,2021
6,Brian Maurer,https://247sports.com/player/brian-maurer-4603...,24,75.0,199,88,"West Port (Ocala, FL)",Ocala,FL,Power 4 Starter,...,1.350000,0.315000,3.350000,4.350000,3.125000,10.000000,34.950000,74.300000,2,2019
7,Brock Glenn,https://247sports.com/player/brock-glenn-46114...,21,74.0,195,90,"Lausanne Collegiate School (Covington, TN)",Covington,TN,4-7 Round - Day 3,...,7.833333,0.370000,2.766667,5.533333,5.863333,11.766667,49.466667,114.200000,3,2023
8,Brock Vandagriff,https://247sports.com/player/brock-vandagriff-...,5,75.0,205,97,"Prince Avenue Christian School (Bogart, GA)",Bogart,GA,Second Round,...,3.700000,0.000000,0.000000,3.066667,3.796667,13.800000,6.866667,60.100000,3,2021
9,Bryce Young,https://247sports.com/player/bryce-young-93127/,1,71.0,183,101,"Mater Dei (Santa Ana, CA)",Santa Ana,CA,Round 1 - Top 10,...,7.166667,0.296667,0.866667,8.266667,9.300000,12.966667,207.200000,154.800000,3,2020


Perect, now let's create a column that decrements it by 1

In [309]:
# Create a new column for the year before enrollment
players['year_before_enrollment'] = players['recruiting_class'] - 1

In [310]:
# Check the results
players

,player,url,rank,height,weight,composite_rating,school_info,city,state,draft_projection,...,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,Counts,recruiting_class,year_before_enrollment
0,Arch Manning,https://247sports.com/player/arch-manning-4608...,1,76.0,215,100,"Isidore Newman (New Orleans, LA)",New Orleans,LA,Round 1 - Top 10,...,0.246667,1.300000,8.066667,8.590000,14.400000,117.400000,139.766667,3,2023,2022
1,Athan Kaliakmanis,https://247sports.com/player/athan-kaliakmanis...,26,76.0,205,90,"Antioch (Antioch, IL)",Antioch,IL,4-7 Round - Day 3,...,0.555000,3.350000,7.400000,6.635000,13.800000,119.600000,121.300000,2,2021,2020
2,Austin Novosad,https://247sports.com/player/austin-novosad-46...,13,75.0,185,94,"Dripping Springs (Dripping Springs, TX)",Dripping Springs,TX,2-3 Round - Day 2,...,0.000000,0.000000,9.400000,9.410000,16.433333,19.866667,156.300000,3,2023,2022
3,Avery Johnson,https://247sports.com/player/avery-johnson-461...,9,74.0,175,95,"Maize (Maize, KS)",Maize,KS,4-7 Round - Day 3,...,0.423333,1.500000,7.200000,7.816667,12.366667,155.766667,136.966667,3,2023,2022
4,Beau Allen,https://247sports.com/player/beau-allen-46039461/,64,74.0,203,85,"Lexington Catholic (Lexington, KY)",Lexington,KY,Power 4 Starter,...,0.000000,0.000000,6.700000,6.690000,12.400000,25.350000,111.000000,2,2020,2019
5,Behren Morton,https://247sports.com/player/behren-morton-460...,11,74.0,185,94,"Eastland (Eastland, TX)",Eastland,TX,4-7 Round - Day 3,...,0.528000,1.960000,5.640000,5.612000,9.060000,161.880000,113.960000,5,2021,2020
6,Brian Maurer,https://247sports.com/player/brian-maurer-4603...,24,75.0,199,88,"West Port (Ocala, FL)",Ocala,FL,Power 4 Starter,...,0.315000,3.350000,4.350000,3.125000,10.000000,34.950000,74.300000,2,2019,2018
7,Brock Glenn,https://247sports.com/player/brock-glenn-46114...,21,74.0,195,90,"Lausanne Collegiate School (Covington, TN)",Covington,TN,4-7 Round - Day 3,...,0.370000,2.766667,5.533333,5.863333,11.766667,49.466667,114.200000,3,2023,2022
8,Brock Vandagriff,https://247sports.com/player/brock-vandagriff-...,5,75.0,205,97,"Prince Avenue Christian School (Bogart, GA)",Bogart,GA,Second Round,...,0.000000,0.000000,3.066667,3.796667,13.800000,6.866667,60.100000,3,2021,2020
9,Bryce Young,https://247sports.com/player/bryce-young-93127/,1,71.0,183,101,"Mater Dei (Santa Ana, CA)",Santa Ana,CA,Round 1 - Top 10,...,0.296667,0.866667,8.266667,9.300000,12.966667,207.200000,154.800000,3,2020,2019


Finally, lets begin merging this with the team stats and standings datasets:

In [311]:
# read in both datasets
team_stats = pd.read_excel(r'../Data/Team_Stats_2018_2025.xlsx')

team_standings = pd.read_excel(r'../Data/Team_Standings_2018_2025.xlsx')

In [312]:
team_stats.columns

Index(['Rk', 'School', 'G', 'Pts', 'Pass_Cmp', 'Pass_Att', 'Pass_Pct',
       'Pass_Yds', 'Pass_TD', 'Rush_Att', 'Rush_Yds', 'Rush_Avg', 'Rush_TD',
       'Off_Plays', 'Off_Yds', 'Off_Avg', 'FD_Pass', 'FD_Rush', 'FD_Pen', 'FD',
       'Pen_No.', 'Pen_Yds', 'Fum', 'Int', 'Tot', 'Season'],
      dtype='str')

In [313]:
# Merge team stats with players
players = players.merge(team_stats, left_on=['committed_school', 'year_before_enrollment'], right_on=['School', 'Season'], how='inner')

In [314]:
# Check that no rows were dropped in the merge
players.shape

(48, 85)

In [315]:
# Merge team standings with players
players = players.merge(team_standings, left_on=['committed_school', 'year_before_enrollment'], right_on=['School', 'Season'], how='inner')

In [316]:
# Check that no rows were dropped in the merge
players.shape

(48, 103)

In [317]:
players

,player,url,rank,height,weight,composite_rating,school_info,city,state,draft_projection,...,Conf_Pct,PPG_Off,PPG_Def,SRS,SOS,AP_Pre,AP_High,AP_Rank,Notes,Season_y
0,Arch Manning,https://247sports.com/player/arch-manning-4608...,1,76.0,215,100,"Isidore Newman (New Orleans, LA)",New Orleans,LA,Round 1 - Top 10,...,0.667,34.5,21.6,15.92,8.16,NaN,18.0,25.0,NaN,2022
1,Athan Kaliakmanis,https://247sports.com/player/athan-kaliakmanis...,26,76.0,205,90,"Antioch (Antioch, IL)",Antioch,IL,4-7 Round - Day 3,...,0.429,27.3,30.1,-1.32,2.10,19.0,19.0,NaN,NaN,2020
2,Austin Novosad,https://247sports.com/player/austin-novosad-46...,13,75.0,185,94,"Dripping Springs (Dripping Springs, TX)",Dripping Springs,TX,2-3 Round - Day 2,...,0.778,38.8,27.4,13.44,3.67,11.0,6.0,15.0,NaN,2022
3,Avery Johnson,https://247sports.com/player/avery-johnson-461...,9,74.0,175,95,"Maize (Maize, KS)",Maize,KS,4-7 Round - Day 3,...,0.778,32.3,21.9,16.16,7.95,NaN,11.0,14.0,NaN,2022
4,Beau Allen,https://247sports.com/player/beau-allen-46039461/,64,74.0,203,85,"Lexington Catholic (Lexington, KY)",Lexington,KY,Power 4 Starter,...,0.375,27.2,19.3,5.92,0.15,NaN,NaN,NaN,NaN,2019
5,Behren Morton,https://247sports.com/player/behren-morton-460...,11,74.0,185,94,"Eastland (Eastland, TX)",Eastland,TX,4-7 Round - Day 3,...,0.333,29.1,36.7,-1.67,3.53,NaN,NaN,NaN,NaN,2020
6,Brian Maurer,https://247sports.com/player/brian-maurer-4603...,24,75.0,199,88,"West Port (Ocala, FL)",Ocala,FL,Power 4 Starter,...,0.250,22.8,27.9,0.39,6.05,NaN,NaN,NaN,NaN,2018
7,Brock Glenn,https://247sports.com/player/brock-glenn-46114...,21,74.0,195,90,"Lausanne Collegiate School (Covington, TN)",Covington,TN,4-7 Round - Day 3,...,0.625,36.1,20.6,14.77,3.38,NaN,11.0,11.0,NaN,2022
8,Brock Vandagriff,https://247sports.com/player/brock-vandagriff-...,5,75.0,205,97,"Prince Avenue Christian School (Bogart, GA)",Bogart,GA,Second Round,...,0.778,32.3,20.0,19.33,8.53,4.0,3.0,7.0,NaN,2020
9,Bryce Young,https://247sports.com/player/bryce-young-93127/,1,71.0,183,101,"Mater Dei (Santa Ana, CA)",Santa Ana,CA,Round 1 - Top 10,...,0.750,47.2,18.6,21.11,2.81,2.0,1.0,8.0,NaN,2019


### Now that we have completed this process, let's clean up the data a bit

First let's remove duplicate columns

In [318]:
# Remove duplicate/ redundant/ not useful columns
players = players.drop(columns=['Player', 'Team', 'Conf_x', 'Rk_x', 'School_x', 'Season_x', 'Rk_y', 'School_y', 'Conf_y', 'Notes', 'Season_y'])

In [319]:
players

,player,url,rank,height,weight,composite_rating,school_info,city,state,draft_projection,...,Conf_W,Conf_L,Conf_Pct,PPG_Off,PPG_Def,SRS,SOS,AP_Pre,AP_High,AP_Rank
0,Arch Manning,https://247sports.com/player/arch-manning-4608...,1,76.0,215,100,"Isidore Newman (New Orleans, LA)",New Orleans,LA,Round 1 - Top 10,...,6.0,3.0,0.667,34.5,21.6,15.92,8.16,NaN,18.0,25.0
1,Athan Kaliakmanis,https://247sports.com/player/athan-kaliakmanis...,26,76.0,205,90,"Antioch (Antioch, IL)",Antioch,IL,4-7 Round - Day 3,...,3.0,4.0,0.429,27.3,30.1,-1.32,2.10,19.0,19.0,NaN
2,Austin Novosad,https://247sports.com/player/austin-novosad-46...,13,75.0,185,94,"Dripping Springs (Dripping Springs, TX)",Dripping Springs,TX,2-3 Round - Day 2,...,7.0,2.0,0.778,38.8,27.4,13.44,3.67,11.0,6.0,15.0
3,Avery Johnson,https://247sports.com/player/avery-johnson-461...,9,74.0,175,95,"Maize (Maize, KS)",Maize,KS,4-7 Round - Day 3,...,7.0,2.0,0.778,32.3,21.9,16.16,7.95,NaN,11.0,14.0
4,Beau Allen,https://247sports.com/player/beau-allen-46039461/,64,74.0,203,85,"Lexington Catholic (Lexington, KY)",Lexington,KY,Power 4 Starter,...,3.0,5.0,0.375,27.2,19.3,5.92,0.15,NaN,NaN,NaN
5,Behren Morton,https://247sports.com/player/behren-morton-460...,11,74.0,185,94,"Eastland (Eastland, TX)",Eastland,TX,4-7 Round - Day 3,...,3.0,6.0,0.333,29.1,36.7,-1.67,3.53,NaN,NaN,NaN
6,Brian Maurer,https://247sports.com/player/brian-maurer-4603...,24,75.0,199,88,"West Port (Ocala, FL)",Ocala,FL,Power 4 Starter,...,2.0,6.0,0.250,22.8,27.9,0.39,6.05,NaN,NaN,NaN
7,Brock Glenn,https://247sports.com/player/brock-glenn-46114...,21,74.0,195,90,"Lausanne Collegiate School (Covington, TN)",Covington,TN,4-7 Round - Day 3,...,5.0,3.0,0.625,36.1,20.6,14.77,3.38,NaN,11.0,11.0
8,Brock Vandagriff,https://247sports.com/player/brock-vandagriff-...,5,75.0,205,97,"Prince Avenue Christian School (Bogart, GA)",Bogart,GA,Second Round,...,7.0,2.0,0.778,32.3,20.0,19.33,8.53,4.0,3.0,7.0
9,Bryce Young,https://247sports.com/player/bryce-young-93127/,1,71.0,183,101,"Mater Dei (Santa Ana, CA)",Santa Ana,CA,Round 1 - Top 10,...,6.0,2.0,0.750,47.2,18.6,21.11,2.81,2.0,1.0,8.0


Next, let's rename certain columns for better clarity

In [320]:
# rename columns to be more descriptive and consistent
players = players.rename(
    columns={
    'rank': 'recruiting_rank',
    'school_info': 'high_school',
    'Season_x': 'Season',
    'G_x': 'games_played_player_after_enrollment',
    'Cmp': 'completions_player_after_enrollment',
    'Att': 'attempts_player_after_enrollment',
    'Cmp%': 'completion_pct_player_after_enrollment',
    'Yds': 'yards_player_after_enrollment',
    'TD': 'touchdowns_player_after_enrollment',
    'TD%': 'touchdown_pct_player_after_enrollment',
    'Int_x': 'interceptions_player_after_enrollment',
    'Int%': 'interception_pct_player_after_enrollment',
    'Y/A': 'yards_per_attempt_player_after_enrollment',
    'AY/A': 'adjusted_yards_per_attempt_player_after_enrollment',
    'Y/C': 'yards_per_completion_player_after_enrollment',
    'Y/G': 'yards_per_game_player_after_enrollment',
    'Rate': 'passing_rating_player_after_enrollment',
    'Counts': 'seasons_played_player_after_enrollment',
    'G_y': 'games_played_team_before_enrollment',
    'Pts': 'ppg_team_before_enrollment',
    'Pass_Cmp': 'passing_completions_team_before_enrollment',
    'Pass_Att': 'passing_attempts_team_before_enrollment',
    'Pass_Pct': 'passing_pct_team_before_enrollment',
    'Pass_Yds': 'passing_yards_team_before_enrollment',
    'Pass_TD': 'passing_touchdowns_team_before_enrollment',
    'Rush_Att': 'rushing_attempts_team_before_enrollment',
    'Rush_Yds': 'rushing_yards_team_before_enrollment',
    'Rush_Avg': 'rushing_average_team_before_enrollment',
    'Rush_TD': 'rushing_touchdowns_team_before_enrollment',
    'Off_Plays': 'offensive_plays_team_before_enrollment',
    'Off_Yds': 'offensive_yards_team_before_enrollment',
    'Off_Avg': 'offensive_average_team_before_enrollment',
    'FD_Pass': 'first_down_passes_team_before_enrollment',
    'FD_Rush': 'first_down_rushes_team_before_enrollment',
    'FD_Pen': 'first_down_penalties_team_before_enrollment',
    'FD': 'first_downs_team_before_enrollment',
    'Pen_No.': 'penalty_counts_team_before_enrollment',
    'Pen_Yds': 'penalty_yards_team_before_enrollment',
    'Fum': 'fumbles_team_before_enrollment',
    'Int_y': 'interceptions_team_before_enrollment',
    'Tot': 'total_turnovers_team_before_enrollment',
    'Ovr_W': 'overall_wins_team_before_enrollment',
    'Ovr_L': 'overall_losses_team_before_enrollment',
    'Ovr_Pct': 'overall_win_pct_team_before_enrollment',
    'Conf_W': 'conference_wins_team_before_enrollment',
    'Conf_L': 'conference_losses_team_before_enrollment',
    'Conf_Pct': 'conference_win_pct_team_before_enrollment',
    'PPG_Off': 'ppg_offense_team_before_enrollment',
    'PPG_Def': 'ppg_defense_team_before_enrollment',
    'SRS': 'srs_team_before_enrollment',
    'SOS': 'sos_team_before_enrollment',
    'AP_Pre': 'ap_pre_team_before_enrollment',
    'AP_High': 'ap_high_team_before_enrollment',
    'AP_Rank': 'ap_rank_team_before_enrollment',
    'Accuracy': 'Accuracy_sentiment_score',
    'Arm Strength': 'Arm_Strength_sentiment_score',
    'Mobility & Athleticism': 'Mobility_and_Athleticism_sentiment_score',
    'NFL / Draft Projection': 'NFL_Draft_Projection_sentiment_score',
    'Pocket Presence': 'Pocket_Presence_sentiment_score',
    'Size & Physical': 'Size_and_Physical_sentiment_score'
    })

Now lets make do a bit more feature engineering:


- create is top ten ranked dummy columns for the following: 'ap_high_team_before_enrollment', 'ap_rank_team_before_enrollment'

- touchdown to interception ratio for the player (this will be the target variable)

In [321]:
# Create column that accesses if player was on a team that finished in the AP top 10 in the year before their enrollment
players['final_ranking_team_AP_top_10_before_enrollment'] = players['ap_rank_team_before_enrollment'].apply(lambda x: 1 if x <= 10 else 0)

# Create column that accesses if player was on a team that got into the AP top 10 at least once during the year before their enrollment
players['highest_ranking_team_AP_top_10_before_enrollment'] = players['ap_high_team_before_enrollment'].apply(lambda x: 1 if x <= 10 else 0)

In [322]:
players

,player,url,recruiting_rank,height,weight,composite_rating,high_school,city,state,draft_projection,...,conference_win_pct_team_before_enrollment,ppg_offense_team_before_enrollment,ppg_defense_team_before_enrollment,srs_team_before_enrollment,sos_team_before_enrollment,ap_pre_team_before_enrollment,ap_high_team_before_enrollment,ap_rank_team_before_enrollment,final_ranking_team_AP_top_10_before_enrollment,highest_ranking_team_AP_top_10_before_enrollment
0,Arch Manning,https://247sports.com/player/arch-manning-4608...,1,76.0,215,100,"Isidore Newman (New Orleans, LA)",New Orleans,LA,Round 1 - Top 10,...,0.667,34.5,21.6,15.92,8.16,NaN,18.0,25.0,0,0
1,Athan Kaliakmanis,https://247sports.com/player/athan-kaliakmanis...,26,76.0,205,90,"Antioch (Antioch, IL)",Antioch,IL,4-7 Round - Day 3,...,0.429,27.3,30.1,-1.32,2.10,19.0,19.0,NaN,0,0
2,Austin Novosad,https://247sports.com/player/austin-novosad-46...,13,75.0,185,94,"Dripping Springs (Dripping Springs, TX)",Dripping Springs,TX,2-3 Round - Day 2,...,0.778,38.8,27.4,13.44,3.67,11.0,6.0,15.0,0,1
3,Avery Johnson,https://247sports.com/player/avery-johnson-461...,9,74.0,175,95,"Maize (Maize, KS)",Maize,KS,4-7 Round - Day 3,...,0.778,32.3,21.9,16.16,7.95,NaN,11.0,14.0,0,0
4,Beau Allen,https://247sports.com/player/beau-allen-46039461/,64,74.0,203,85,"Lexington Catholic (Lexington, KY)",Lexington,KY,Power 4 Starter,...,0.375,27.2,19.3,5.92,0.15,NaN,NaN,NaN,0,0
5,Behren Morton,https://247sports.com/player/behren-morton-460...,11,74.0,185,94,"Eastland (Eastland, TX)",Eastland,TX,4-7 Round - Day 3,...,0.333,29.1,36.7,-1.67,3.53,NaN,NaN,NaN,0,0
6,Brian Maurer,https://247sports.com/player/brian-maurer-4603...,24,75.0,199,88,"West Port (Ocala, FL)",Ocala,FL,Power 4 Starter,...,0.250,22.8,27.9,0.39,6.05,NaN,NaN,NaN,0,0
7,Brock Glenn,https://247sports.com/player/brock-glenn-46114...,21,74.0,195,90,"Lausanne Collegiate School (Covington, TN)",Covington,TN,4-7 Round - Day 3,...,0.625,36.1,20.6,14.77,3.38,NaN,11.0,11.0,0,0
8,Brock Vandagriff,https://247sports.com/player/brock-vandagriff-...,5,75.0,205,97,"Prince Avenue Christian School (Bogart, GA)",Bogart,GA,Second Round,...,0.778,32.3,20.0,19.33,8.53,4.0,3.0,7.0,1,1
9,Bryce Young,https://247sports.com/player/bryce-young-93127/,1,71.0,183,101,"Mater Dei (Santa Ana, CA)",Santa Ana,CA,Round 1 - Top 10,...,0.750,47.2,18.6,21.11,2.81,2.0,1.0,8.0,1,1


Now, lets create the touchdown/ interception ratio

In [323]:
# Create touchdowns to interception ratio column for player after enrollment
players['TD/INT_Ratio_player_after_enrollment'] = players['touchdowns_player_after_enrollment'] / players['interceptions_player_after_enrollment']

In [324]:
# look at the results
players[['player', 'committed_school', 'touchdowns_player_after_enrollment','interceptions_player_after_enrollment', 'TD/INT_Ratio_player_after_enrollment']]

,player,committed_school,touchdowns_player_after_enrollment,interceptions_player_after_enrollment,TD/INT_Ratio_player_after_enrollment
0,Arch Manning,Texas,0.966667,0.246667,3.918919
1,Athan Kaliakmanis,Minnesota,0.720000,0.555000,1.297297
2,Austin Novosad,Oregon,0.000000,0.000000,NaN
3,Avery Johnson,Kansas State,1.350000,0.423333,3.188976
4,Beau Allen,Kentucky,0.000000,0.000000,NaN
5,Behren Morton,Texas Tech,1.272000,0.528000,2.409091
6,Brian Maurer,Tennessee,0.125000,0.315000,0.396825
7,Brock Glenn,Florida State,0.356667,0.370000,0.963964
8,Brock Vandagriff,Georgia,0.083333,0.000000,inf
9,Bryce Young,Alabama,1.976667,0.296667,6.662921


Note: few players that do not have recorded interceptions.

After further analysis, these players did not get much playing time at their respective schools and are likely not good data points anyway. Therefore, they will be dropped.

In [325]:
# Drop rows where TD/INT ratio is infinite (indicating 0 interceptions) or null (indicating 0 touchdowns and 0 interceptions)
players = players.replace([np.inf, -np.inf], np.nan).dropna(subset=['TD/INT_Ratio_player_after_enrollment'])

In [326]:
# look at the results
players[['player', 'committed_school', 'touchdowns_player_after_enrollment','interceptions_player_after_enrollment', 'TD/INT_Ratio_player_after_enrollment']]

,player,committed_school,touchdowns_player_after_enrollment,interceptions_player_after_enrollment,TD/INT_Ratio_player_after_enrollment
0,Arch Manning,Texas,0.966667,0.246667,3.918919
1,Athan Kaliakmanis,Minnesota,0.720000,0.555000,1.297297
3,Avery Johnson,Kansas State,1.350000,0.423333,3.188976
5,Behren Morton,Texas Tech,1.272000,0.528000,2.409091
6,Brian Maurer,Tennessee,0.125000,0.315000,0.396825
7,Brock Glenn,Florida State,0.356667,0.370000,0.963964
9,Bryce Young,Alabama,1.976667,0.296667,6.662921
10,C.J. Stroud,Ohio State,3.410000,0.480000,7.104167
11,Cade Klubnik,Clemson,1.390000,0.480000,2.895833
12,Cade McNamara,Michigan,0.883333,0.253333,3.486842


### Now that those are finished, let's look at missingness.

In [327]:
# Examine missingness in the data
players.isna().sum()

player                                               0
url                                                  0
recruiting_rank                                      0
height                                               0
weight                                               0
                                                    ..
ap_high_team_before_enrollment                      11
ap_rank_team_before_enrollment                      24
final_ranking_team_AP_top_10_before_enrollment       0
highest_ranking_team_AP_top_10_before_enrollment     0
TD/INT_Ratio_player_after_enrollment                 0
Length: 95, dtype: int64

Any of the missingness that is present are in columns that will not be used during the modeling phase. Therefore, let's export the dataset.

In [328]:
# Export the dataframe to a new CSV file
players.to_csv(r'../Data/final_player_data.csv', index=False)